# Etapa 1: Palabras clave para la busqueda de noticias

In [1]:
#!pip install openpyxl

In [2]:
import pandas as pd
import numpy as np

# Crear dataframe
credicorp = pd.DataFrame({
    "CONCEPTOS": [
        "BCP",
        "Credicorp",
        "YAPE",
        "Mibanco",
        "Tarjeta IO",
        "Banco de Crédito",
        "Grupo Romero",
        "Tenpo",
        "Krealo"
    ],
    "INCLUIR": [
        "Perú",
        "Credicorp",
        "YAPE",
        "Mibanco",
        "Tarjeta IO",
        "Banco de Crédito",
        "Grupo Romero",
        "Tenpo",
        "Krealo"
    ]
})

print(credicorp)


          CONCEPTOS           INCLUIR
0               BCP              Perú
1         Credicorp         Credicorp
2              YAPE              YAPE
3           Mibanco           Mibanco
4        Tarjeta IO        Tarjeta IO
5  Banco de Crédito  Banco de Crédito
6      Grupo Romero      Grupo Romero
7             Tenpo             Tenpo
8            Krealo            Krealo


In [3]:
# Agregar comillas a cada valor PARA LA BUSQUEDA DEL NOMBRE COMPLETO
credicorp["CONCEPTOS"] = '"' + credicorp["CONCEPTOS"] + '"'
credicorp["INCLUIR"] = '"' + credicorp["INCLUIR"] + '"'

credicorp

,CONCEPTOS,INCLUIR
0,"""BCP""","""Perú"""
1,"""Credicorp""","""Credicorp"""
2,"""YAPE""","""YAPE"""
3,"""Mibanco""","""Mibanco"""
4,"""Tarjeta IO""","""Tarjeta IO"""
5,"""Banco de Crédito""","""Banco de Crédito"""
6,"""Grupo Romero""","""Grupo Romero"""
7,"""Tenpo""","""Tenpo"""
8,"""Krealo""","""Krealo"""


In [4]:
# Crear el diccionario para almacenar los resultados
diccionario = credicorp.to_dict(orient='records')

# Etapa 2: Búsqueda de noticias en la web

## Etapa 2.1: Conexión a Oracle para extraer fecha máx de noticias ya extraídas

In [ ]:
import time
import pandas as pd
import cx_Oracle

# Credenciales
USER = "user"
PWD  = "password"
EZCONNECT = "ex9m1-scan:####/abcde"  # host:puerto/servicename

def execute_query(query: str, data=None):
    """
    Ejecuta queries en Oracle (SELECT, INSERT, UPDATE, DELETE, etc.)
    - SELECT devuelve un DataFrame.
    - INSERT admite inserciones masivas (usando executemany).
    - Otros comandos se ejecutan y confirman sin retorno.
    """
    start_time = time.time()
    conn = None
    try:
        conn = cx_Oracle.connect(user=USER, password=PWD, dsn=EZCONNECT, encoding="UTF-8")
        cursor = conn.cursor()

        # SELECT
        if query.strip().lower().startswith('select'):
            df = pd.read_sql(query, con=conn)
            elapsed_time = time.time() - start_time
            print(f'Consulta ejecutada en {elapsed_time:.2f} segundos.')
            return df

        # INSERT masivo o individual
        elif query.strip().lower().startswith('insert') and data is not None:
            cursor.executemany(query, data)
            conn.commit()
            elapsed_time = time.time() - start_time
            print(f'Insert ejecutado en {elapsed_time:.2f} segundos.')
            return None

        # Otros comandos (UPDATE, DELETE, CREATE, DROP, etc.)
        else:
            cursor.execute(query)
            conn.commit()
            elapsed_time = time.time() - start_time
            print(f'Query ejecutada en {elapsed_time:.2f} segundos.')
            return None

    except Exception as e:
        print(f"Error al ejecutar la query: {e}")
        if conn:
            conn.rollback()
    
    finally:
        if conn:
            conn.close()


In [ ]:
p = '''
SELECT TO_CHAR(
       TRUNC(MAX(TO_DATE(PUBLISHED_DATE,'DD/MM/YYYY HH24:MI:SS'))),
       'DD/MM/YYYY HH24:MI:SS'
)
FROM NOTICIAS_CREDICORP
'''

resultado = execute_query(p)

fecha = resultado.iloc[0,0]

print(fecha)

## Etapa 2.2: Uso de RSS de Google News

In [7]:
#!pip install feedparser

### Setear la función de búsqueda de noticias a través de google RSS

In [8]:
import feedparser as fp
import pandas as pd
from urllib.parse import quote
from datetime import datetime

# URL del feed RSS de Google News
#rss_url = "https://news.google.com/rss?hl=es&gl=ES&ceid=ES:es"

def search_rss( CONCEPTOS,INCLUIR,start_date, results):
    n = results
    CONCEPTOS = CONCEPTOS
    INCLUIR = INCLUIR
    query = f'''{CONCEPTOS} AND {INCLUIR} '''
    
    # Parámetros de búsqueda
    encoded_query = quote(query)  
    # Codifica el texto para URLs
    rss_url = f"https://news.google.com/rss/search?q={encoded_query}&hl=es&gl=PE&ceid=PE:es"

    # Parsear el feed
    feed = fp.parse(rss_url)
    start_date = datetime.strptime(start_date, '%d/%m/%Y %H:%M:%S')

    # Filtrar las entradas usando comprensión de listas
    sorted_entries = sorted(
        (entry for entry in feed.entries if datetime.strptime(entry.published, '%a, %d %b %Y %H:%M:%S  %Z') > start_date),
        key=lambda entry: datetime.strptime(entry.published, '%a, %d %b %Y %H:%M:%S %Z'),
        reverse=True  # Si quieres ordenarlas de más reciente a más antiguo
    )
    
    top_n_entries = sorted_entries[:n]

    return top_n_entries
    
    #for entry in top_n_entries:
    #    print(entry.title, '\n', entry.published , '\n', '-'*80)
        

**Establecer periodo inicial de búsqueda de noticias**

In [9]:
import random
import time 

results = 10 
contador = 0 
total_concepts = len(diccionario) 
start_date = fecha.strip()

start_time = time.time()

# Aplicar la función search_rss
for entry in diccionario:
    contador += 1       
    CONCEPTOS = entry['CONCEPTOS'] 
    INCLUIR = entry['INCLUIR'] 
    # Llamar a search_rss
    entry['RSS'] = search_rss(CONCEPTOS,INCLUIR,  start_date, results) 
    
    print(f"\rEmpresa {contador} de {total_concepts}: {CONCEPTOS} analizada.{' ' * 100}", end="") 

end_time = time.time()
elapsed_time = end_time - start_time

mins, secs = divmod(elapsed_time, 60)
hours, mins = divmod(mins, 60)

print(f"\n⏱️ Tiempo: {int(hours)}h {int(mins)}m {secs:.2f}s")

Empresa 9 de 9: "Krealo" analizada.                                                                                                              
⏱️ Tiempo: 0h 0m 25.84s


In [10]:
PALABRAS_EXCLUIR = [ "farandula", "espectaculos"]
rows = [] 

for entry in diccionario: 
    CONCEPTOS = entry["CONCEPTOS"] 
    
    for rss in entry["RSS"]: 

        titulo = rss["title"]
        link = rss["link"]
        
        # excluir noticias irrelevantes
        if any(palabra in titulo or palabra in link for palabra in PALABRAS_EXCLUIR):
            continue
            
        published = datetime.strptime(
            rss["published"],
            "%a, %d %b %Y %H:%M:%S %Z"
            ) 
        rows.append({ 
            "Concepts": CONCEPTOS, 
            "Title": rss["title"], 
            "Link": rss["link"], 
            "Published_date": published.strftime("%d/%m/%Y %H:%M:%S") 
        }) 
        
# Crear un DataFrame 
resultados_link_rss = pd.DataFrame(rows)

In [11]:
resultados_link_rss

,Concepts,Title,Link,Published_date
0,"""BCP""",BCP en Estados Unidos: ¿Qué significa la aprob...,https://news.google.com/rss/articles/CBMi_gFBV...,29/04/2026 18:08:03
1,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMixgFBV...,29/04/2026 01:13:57
2,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMi6AFBV...,28/04/2026 21:47:07
3,"""BCP""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00
4,"""YAPE""",Yape y Plin: ¿por qué podrían dejar de usarse ...,https://news.google.com/rss/articles/CBMiqAFBV...,29/04/2026 21:09:00
5,"""YAPE""",Jesús María: delincuente en moto arrebata celu...,https://news.google.com/rss/articles/CBMimAFBV...,28/04/2026 19:32:24
6,"""YAPE""",“Abrí mi Yape y me robaron S/2.000”: así opera...,https://news.google.com/rss/articles/CBMivwFBV...,28/04/2026 04:11:00
7,"""Mibanco""",Mibanco obtuvo calificación de riesgo 'AAA' de...,https://news.google.com/rss/articles/CBMixwFBV...,29/04/2026 15:11:51
8,"""Banco de Crédito""",Villa Clara: realizan especialistas de BANDEC ...,https://news.google.com/rss/articles/CBMivgJBV...,28/04/2026 21:52:51
9,"""Banco de Crédito""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00


In [12]:
#resultados_link_rss.to_excel(".xlsx", index=False)

In [13]:
num_not= len(resultados_link_rss)
num_conc= resultados_link_rss['Concepts'].nunique() 

print(f"Hay {num_not} noticias correspondientes a {num_conc} conceptos")

Hay 10 noticias correspondientes a 4 conceptos


### Transformar el link para la extracción 

In [14]:
#pip install googlenewsdecoder

In [15]:
import time
from googlenewsdecoder import gnewsdecoder

start_time = time.time()


total_news = len(resultados_link_rss)
contador = 0

decoded_urls = []
for url in resultados_link_rss['Link']:
    contador+= 1
    try:
        decoded = gnewsdecoder(url, interval=1)['decoded_url']
    except Exception as e:
        decoded = None  # en caso de error
        print(f"Error con {url}: {e}")
    decoded_urls.append(decoded)
    time.sleep(2)  # <-- espera 1 segundo entre cada request

    # Imprimir el progreso y el nombre de la empresa
    print(f"\rNoticia {contador} de {total_news}:  analizada.{' ' * 100}", end="")
 
resultados_link_rss['URL'] = decoded_urls


end_time = time.time()
elapsed_time = end_time - start_time

mins, secs = divmod(elapsed_time, 60)
hours, mins = divmod(mins, 60)

print(f"\n⏱️ Tiempo total de ejecución: {int(hours)}h {int(mins)}m {secs:.2f}s")

Noticia 10 de 10:  analizada.                                                                                                    
⏱️ Tiempo total de ejecución: 0h 0m 59.22s


In [16]:
resultados_link_rss

,Concepts,Title,Link,Published_date,URL
0,"""BCP""",BCP en Estados Unidos: ¿Qué significa la aprob...,https://news.google.com/rss/articles/CBMi_gFBV...,29/04/2026 18:08:03,https://larepublica.pe/economia/2026/04/28/bcp...
1,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMixgFBV...,29/04/2026 01:13:57,https://gestion.pe/peru/bcp-recibe-aprobacion-...
2,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMi6AFBV...,28/04/2026 21:47:07,https://elcomercio.pe/economia/peru/bcp-recibe...
3,"""BCP""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...
4,"""YAPE""",Yape y Plin: ¿por qué podrían dejar de usarse ...,https://news.google.com/rss/articles/CBMiqAFBV...,29/04/2026 21:09:00,https://gestion.pe/tu-dinero/yape-y-plin-por-q...
5,"""YAPE""",Jesús María: delincuente en moto arrebata celu...,https://news.google.com/rss/articles/CBMimAFBV...,28/04/2026 19:32:24,https://www.atv.pe/noticia/jesus-maria-delincu...
6,"""YAPE""",“Abrí mi Yape y me robaron S/2.000”: así opera...,https://news.google.com/rss/articles/CBMivwFBV...,28/04/2026 04:11:00,https://www.infobae.com/peru/2026/04/28/abri-m...
7,"""Mibanco""",Mibanco obtuvo calificación de riesgo 'AAA' de...,https://news.google.com/rss/articles/CBMixwFBV...,29/04/2026 15:11:51,https://www.larepublica.co/finanzas/mibanco-ob...
8,"""Banco de Crédito""",Villa Clara: realizan especialistas de BANDEC ...,https://news.google.com/rss/articles/CBMivgJBV...,28/04/2026 21:52:51,https://www.cmhw.cu/villa-clara/realizan-espec...
9,"""Banco de Crédito""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...


In [17]:
#resultados_link_largo.to_excel("URL.xlsx", index=False)

## Etapa 2.3: Uso de Web Scrapping

### Web scrapping del contenido de la noticia

In [18]:
#pip install selenium requests beautifulsoup4

In [19]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, random

# CONFIGURACIÓN DEL DRIVER
# ------------------------
def nuevo_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("user-agent=Mozilla/5.0")
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver


# FUNCIÓN DE SCRAPING
# ------------------------
import re

def extraer_noticias(url, driver):
    try:
        driver.get(url)

        # Esperar hasta que haya <p>
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, "p"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")
        contenido = []

        # Palabras o frases indeseadas
        indeseables = ["cookies", "privacidad", "anuncios adecuados","términos", "condiciones", "protección de datos","suscríbete", "newsletter", "publicidad","contenido patrocinado", "compartir", "síguenos"]

        if url.startswith("https://www.gob.pe/"):
            parrafos = soup.find("div", class_="description feed-content")
            if parrafos:
                for p in parrafos.find_all("p"):
                    text = p.get_text(strip=True)
                    if (
                        len(text.split()) > 5
                        and not any(word in text.lower() for word in indeseables)
                        and not (text.count("»") >= 2 and len(text.split()) < 12)
                    ):
                        contenido.append(text)
        else:
            # Buscar contenedores con class que contenga "body"
            contenedores = soup.find_all(
                ["div", "section", "article"], 
                class_=re.compile("body", re.I)
            )

            if contenedores:
                for c in contenedores:
                    for p in c.find_all("p"):
                        text = p.get_text(strip=True)
                        if (
                            len(text.split()) > 5
                            and not any(word in text.lower() for word in indeseables)
                            and not (text.count("»") >= 2 and len(text.split()) < 12)
                        ):
                            contenido.append(text)
            else:
                # Fallback: todos los <p> si no hay "body"
                for p in soup.find_all("p"):
                    text = p.get_text(strip=True)
                    if (
                        len(text.split()) > 5
                        and not p.find_parent(["footer", "nav", "button", "aside", "header", "form"])
                        and not any(word in text.lower() for word in indeseables)
                        and not (text.count("»") >= 2 and len(text.split()) < 12)
                    ):
                        contenido.append(text)

        # Concatenar todo el contenido válido
        texto = " ".join(contenido)

        return texto if texto else None

    except Exception as e:
        print(f"❌ {type(e).__name__} en {url}: {e}")
        return None

**Ejecución**

In [20]:
import time
import random

start_time = time.time()
total_news = len(resultados_link_rss)
contador = 0

driver = nuevo_driver()
contenidos = []

for i, url in enumerate(resultados_link_rss["URL"], start=1):
    contador += 1

    texto = extraer_noticias(url, driver)
    contenidos.append(texto)

    # Reiniciar driver cada 5 links
    if i % 5 == 0:
        driver.quit()
        driver = nuevo_driver()

    time.sleep(random.uniform(2, 5))

    print(f"\rNoticia {contador} de {total_news} analizada.{' ' * 100}", end="")

driver.quit()

end_time = time.time()
elapsed_time = end_time - start_time

mins, secs = divmod(elapsed_time, 60)
hours, mins = divmod(mins, 60)

print(f"\n⏱️ Tiempo : {int(hours)}h {int(mins)}m {secs:.2f}s")

Noticia 2 de 10 analizada.                                                                                                    ❌ TimeoutException en https://elcomercio.pe/economia/peru/bcp-recibe-aprobacion-de-la-reserva-federal-para-aperturar-una-sucursal-bancaria-en-estados-unidos-l-ultimas-noticia/: Message: timeout: Timed out receiving message from renderer: 26.833
  (Session info: chrome=147.0.7727.56)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff69579a8e5+14e45]
	chromedriver!GetHandleVerifier [0x7ff69579a950+14eb0]
	chromedriver!(No symbol) [0x7ff69550d6ed]
	chromedriver!(No symbol) [0x7ff6954f9ec9]
	chromedriver!(No symbol) [0x7ff6954f9bb1]
	chromedriver!(No symbol) [0x7ff6954f77e1]
	chromedriver!(No symbol) [0x7ff6954f811f]
	chromedriver!(No symbol) [0x7ff695507aaa]
	chromedriver!(No symbol) [0x7ff69551e9b7]
	chromedriver!(No symbol) [0x7ff69552610a]
	chromedriver!(No symbol) [0x7ff6954f88e1]
	chromedriver!(No symbol) [0x7ff69551e6e5]
	chromedriver!(No symbol) [0x7ff6955b4

In [21]:
# Guardar contenido en el bloque actual
resultados_link_rss["Content"] = contenidos
resultados_link_rss

,Concepts,Title,Link,Published_date,URL,Content
0,"""BCP""",BCP en Estados Unidos: ¿Qué significa la aprob...,https://news.google.com/rss/articles/CBMi_gFBV...,29/04/2026 18:08:03,https://larepublica.pe/economia/2026/04/28/bcp...,La autorización otorgada por laReserva Federal...
1,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMixgFBV...,29/04/2026 01:13:57,https://gestion.pe/peru/bcp-recibe-aprobacion-...,None
2,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMi6AFBV...,28/04/2026 21:47:07,https://elcomercio.pe/economia/peru/bcp-recibe...,None
3,"""BCP""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...
4,"""YAPE""",Yape y Plin: ¿por qué podrían dejar de usarse ...,https://news.google.com/rss/articles/CBMiqAFBV...,29/04/2026 21:09:00,https://gestion.pe/tu-dinero/yape-y-plin-por-q...,None
5,"""YAPE""",Jesús María: delincuente en moto arrebata celu...,https://news.google.com/rss/articles/CBMimAFBV...,28/04/2026 19:32:24,https://www.atv.pe/noticia/jesus-maria-delincu...,Undelincuente en motorobó el celular de una mu...
6,"""YAPE""",“Abrí mi Yape y me robaron S/2.000”: así opera...,https://news.google.com/rss/articles/CBMivwFBV...,28/04/2026 04:11:00,https://www.infobae.com/peru/2026/04/28/abri-m...,Lainseguridad ciudadana en Limasigue evolucion...
7,"""Mibanco""",Mibanco obtuvo calificación de riesgo 'AAA' de...,https://news.google.com/rss/articles/CBMixwFBV...,29/04/2026 15:11:51,https://www.larepublica.co/finanzas/mibanco-ob...,"Nancy Tueros, presidenta de Mibanco Colombia h..."
8,"""Banco de Crédito""",Villa Clara: realizan especialistas de BANDEC ...,https://news.google.com/rss/articles/CBMivgJBV...,28/04/2026 21:52:51,https://www.cmhw.cu/villa-clara/realizan-espec...,Para contribuir a la educación financiera en l...
9,"""Banco de Crédito""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...


In [22]:
total_not = len(resultados_link_rss)
noticia_con_contenido= resultados_link_rss['Content'].notna().sum()

print("RESUMEN DE LA EXTRACCIÓN DE CONTENIDO")
print(f"📄  Se logró extraer el contenido de {noticia_con_contenido} noticias por web scrapping, de las {total_not} encontradas por G. News")

RESUMEN DE LA EXTRACCIÓN DE CONTENIDO
📄  Se logró extraer el contenido de 7 noticias por web scrapping, de las 10 encontradas por G. News


In [23]:
#resultados_link_rss.to_excel("credicorp_sinlimpiar.xlsx", index=False)

## Etapa 2.4: Tabla final para dashboard

#### Corrección de mensaje de error propio de la página

In [24]:
frases_invalidas = [
    "Todos los derechos reservados",
    "This error was generated by Mod_Security",
    "Verifying you are human",
    "its security certificate is not trusted by your computer's operating system",
    "Regístrese y obtenga5 artículos gratisal mes y el boletín informativo",
    "Si quieres sacarle el mayor partido a nuestro sitio,regístrate",
    "The requested URL was not found on this server",
    "This website is using a security service to protect itself",
    "this site is currently experiencing technical difficulties",
    "this website uses a security service to protect",
    "the web page is not displaying.",
    "The server encountered an internal error",
    "We appreciate your understanding as we work to protect our",
    "it may have moved permanently to a new web address",
    "This may be caused by a misconfiguration or an attacker",
    "You do not have access",
    "Complete the security check",
    "Los datos compartidos se usarán para guardar sus preferencias en este sitio"
]

patron = "|".join(map(re.escape, frases_invalidas))

# Filas con frases inválidas
mask_invalidas = resultados_link_rss["Content"].str.contains(
    patron, case=False, na=False
)
n_invalidas = mask_invalidas.sum()
print(f"📄  De los {noticia_con_contenido} contenidos extraídos, {n_invalidas} son érroneos por protección y naturaleza de la página (serán reemplazados por el título)")


📄  De los 7 contenidos extraídos, 0 son érroneos por protección y naturaleza de la página (serán reemplazados por el título)


#### Para los casos con extracción del cuerpo erronea por naturaleza de la web.

In [25]:
import numpy as np

# Reemplazar el contenido por el título en las filas con frases inválidas
resultados_link_rss.loc[mask_invalidas, "Content"] = np.nan
resultados_link_rss

,Concepts,Title,Link,Published_date,URL,Content
0,"""BCP""",BCP en Estados Unidos: ¿Qué significa la aprob...,https://news.google.com/rss/articles/CBMi_gFBV...,29/04/2026 18:08:03,https://larepublica.pe/economia/2026/04/28/bcp...,La autorización otorgada por laReserva Federal...
1,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMixgFBV...,29/04/2026 01:13:57,https://gestion.pe/peru/bcp-recibe-aprobacion-...,None
2,"""BCP""",BCP recibe aprobación de la Reserva Federal pa...,https://news.google.com/rss/articles/CBMi6AFBV...,28/04/2026 21:47:07,https://elcomercio.pe/economia/peru/bcp-recibe...,None
3,"""BCP""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...
4,"""YAPE""",Yape y Plin: ¿por qué podrían dejar de usarse ...,https://news.google.com/rss/articles/CBMiqAFBV...,29/04/2026 21:09:00,https://gestion.pe/tu-dinero/yape-y-plin-por-q...,None
5,"""YAPE""",Jesús María: delincuente en moto arrebata celu...,https://news.google.com/rss/articles/CBMimAFBV...,28/04/2026 19:32:24,https://www.atv.pe/noticia/jesus-maria-delincu...,Undelincuente en motorobó el celular de una mu...
6,"""YAPE""",“Abrí mi Yape y me robaron S/2.000”: así opera...,https://news.google.com/rss/articles/CBMivwFBV...,28/04/2026 04:11:00,https://www.infobae.com/peru/2026/04/28/abri-m...,Lainseguridad ciudadana en Limasigue evolucion...
7,"""Mibanco""",Mibanco obtuvo calificación de riesgo 'AAA' de...,https://news.google.com/rss/articles/CBMixwFBV...,29/04/2026 15:11:51,https://www.larepublica.co/finanzas/mibanco-ob...,"Nancy Tueros, presidenta de Mibanco Colombia h..."
8,"""Banco de Crédito""",Villa Clara: realizan especialistas de BANDEC ...,https://news.google.com/rss/articles/CBMivgJBV...,28/04/2026 21:52:51,https://www.cmhw.cu/villa-clara/realizan-espec...,Para contribuir a la educación financiera en l...
9,"""Banco de Crédito""",La Reserva Federal de Estados Unidos aprueba a...,https://news.google.com/rss/articles/CBMiwgFBV...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...


#### Columnas de interés

In [26]:
resultados_link_rss['Concepts'] = resultados_link_rss['Concepts'].str.strip('"').str.strip()

In [27]:
resultados_link_rss = resultados_link_rss.drop(columns=['Link'])

#### Para los casos sin extracción del cuerpo, solo se analizará el título

In [28]:
resultados_link_rss['Content'] = resultados_link_rss['Content'].fillna(resultados_link_rss['Title'])
resultados_link_rss

,Concepts,Title,Published_date,URL,Content
0,BCP,BCP en Estados Unidos: ¿Qué significa la aprob...,29/04/2026 18:08:03,https://larepublica.pe/economia/2026/04/28/bcp...,La autorización otorgada por laReserva Federal...
1,BCP,BCP recibe aprobación de la Reserva Federal pa...,29/04/2026 01:13:57,https://gestion.pe/peru/bcp-recibe-aprobacion-...,BCP recibe aprobación de la Reserva Federal pa...
2,BCP,BCP recibe aprobación de la Reserva Federal pa...,28/04/2026 21:47:07,https://elcomercio.pe/economia/peru/bcp-recibe...,BCP recibe aprobación de la Reserva Federal pa...
3,BCP,La Reserva Federal de Estados Unidos aprueba a...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...
4,YAPE,Yape y Plin: ¿por qué podrían dejar de usarse ...,29/04/2026 21:09:00,https://gestion.pe/tu-dinero/yape-y-plin-por-q...,Yape y Plin: ¿por qué podrían dejar de usarse ...
5,YAPE,Jesús María: delincuente en moto arrebata celu...,28/04/2026 19:32:24,https://www.atv.pe/noticia/jesus-maria-delincu...,Undelincuente en motorobó el celular de una mu...
6,YAPE,“Abrí mi Yape y me robaron S/2.000”: así opera...,28/04/2026 04:11:00,https://www.infobae.com/peru/2026/04/28/abri-m...,Lainseguridad ciudadana en Limasigue evolucion...
7,Mibanco,Mibanco obtuvo calificación de riesgo 'AAA' de...,29/04/2026 15:11:51,https://www.larepublica.co/finanzas/mibanco-ob...,"Nancy Tueros, presidenta de Mibanco Colombia h..."
8,Banco de Crédito,Villa Clara: realizan especialistas de BANDEC ...,28/04/2026 21:52:51,https://www.cmhw.cu/villa-clara/realizan-espec...,Para contribuir a la educación financiera en l...
9,Banco de Crédito,La Reserva Federal de Estados Unidos aprueba a...,28/04/2026 14:22:00,https://www.infobae.com/peru/2026/04/25/la-res...,LaReserva Federal de Estados Unidosdio luz ver...


In [29]:
MAX_LEN = 4000
SUFFIX = " [...]"

resultados_link_rss["Content"] = resultados_link_rss["Content"].apply(
    lambda x: x[:MAX_LEN - len(SUFFIX)] + SUFFIX if isinstance(x, str) and len(x) > MAX_LEN else x
)

In [30]:
# resultados_link_rss.to_excel(".xlsx", index=False)

# Etapa 3: Subir la información al esquema

In [ ]:
import time
import pandas as pd
import cx_Oracle

# Credenciales
USER = "user"
PWD  = "password"
EZCONNECT = "ex9m1-scan:####/abdce"  # host:puerto/servicename

def execute_query(query: str, data=None):
    """
    Ejecuta queries en Oracle (SELECT, INSERT, UPDATE, DELETE, etc.)
    - SELECT devuelve un DataFrame.
    - INSERT admite inserciones masivas (usando executemany).
    - Otros comandos se ejecutan y confirman sin retorno.
    """
    start_time = time.time()
    conn = None
    try:
        conn = cx_Oracle.connect(user=USER, password=PWD, dsn=EZCONNECT, encoding="UTF-8")
        cursor = conn.cursor()

        # SELECT
        if query.strip().lower().startswith('select'):
            df = pd.read_sql(query, con=conn)
            elapsed_time = time.time() - start_time
            print(f'Consulta ejecutada en {elapsed_time:.2f} segundos.')
            return df

        # INSERT masivo o individual
        elif query.strip().lower().startswith('insert') and data is not None:
            cursor.executemany(query, data)
            conn.commit()
            elapsed_time = time.time() - start_time
            print(f'Insert ejecutado en {elapsed_time:.2f} segundos.')
            return None

        # Otros comandos (UPDATE, DELETE, CREATE, DROP, etc.)
        else:
            cursor.execute(query)
            conn.commit()
            elapsed_time = time.time() - start_time
            print(f'Query ejecutada en {elapsed_time:.2f} segundos.')
            return None

    except Exception as e:
        print(f"Error al ejecutar la query: {e}")
        if conn:
            conn.rollback()
    
    finally:
        if conn:
            conn.close()


In [ ]:
query_existentes = """
SELECT DISTINCT CONCEPT, URL as URL_2
FROM NOTICIAS_CREDICORP
"""
existentes = execute_query(query_existentes)

In [33]:
df_filtrado = resultados_link_rss.merge( existentes,
    left_on=["Concepts", "URL"],
    right_on=["CONCEPT", "URL_2"],
    how="left", indicator=True )

df_filtrado = df_filtrado[df_filtrado["_merge"] == "left_only"]
df_filtrado = df_filtrado.drop(columns=["CONCEPT", "URL_2", "_merge"],errors="ignore")
df_filtrado

,Concepts,Title,Published_date,URL,Content
0,BCP,BCP en Estados Unidos: ¿Qué significa la aprob...,29/04/2026 18:08:03,https://larepublica.pe/economia/2026/04/28/bcp...,La autorización otorgada por laReserva Federal...
1,BCP,BCP recibe aprobación de la Reserva Federal pa...,29/04/2026 01:13:57,https://gestion.pe/peru/bcp-recibe-aprobacion-...,BCP recibe aprobación de la Reserva Federal pa...
2,BCP,BCP recibe aprobación de la Reserva Federal pa...,28/04/2026 21:47:07,https://elcomercio.pe/economia/peru/bcp-recibe...,BCP recibe aprobación de la Reserva Federal pa...
4,YAPE,Yape y Plin: ¿por qué podrían dejar de usarse ...,29/04/2026 21:09:00,https://gestion.pe/tu-dinero/yape-y-plin-por-q...,Yape y Plin: ¿por qué podrían dejar de usarse ...
7,Mibanco,Mibanco obtuvo calificación de riesgo 'AAA' de...,29/04/2026 15:11:51,https://www.larepublica.co/finanzas/mibanco-ob...,"Nancy Tueros, presidenta de Mibanco Colombia h..."
8,Banco de Crédito,Villa Clara: realizan especialistas de BANDEC ...,28/04/2026 21:52:51,https://www.cmhw.cu/villa-clara/realizan-espec...,Para contribuir a la educación financiera en l...


In [34]:
# convertir dataframe a lista de tuplas
resultados_noticias_list = list(df_filtrado.itertuples(index=False, name=None))
resultados_noticias_list

[('BCP',
  'BCP en Estados Unidos: ¿Qué significa la aprobación de la Reserva Federal y cómo beneficiará a clientes y empresas? - La República',
  '29/04/2026 18:08:03',
  'https://larepublica.pe/economia/2026/04/28/bcp-en-estados-unidos-que-significa-la-aprobacion-de-la-reserva-federal-y-como-beneficiara-a-clientes-y-empresas-hnews-1346324',
  'La autorización otorgada por laReserva Federal de Estados Unidos—equivalente alBanco Central de Reserva del Perú (BCRP)— para que elBanco de Crédito del Perú (BCP)abra una sucursal enMiamimarca un nuevo paso en la proyección internacional de labanca peruana. Esta expansión busca reforzar loslazos financieros entre ambos paísesy agilizar las operaciones de clientes y empresas con vínculos comerciales o familiares en el extranjero. En declaraciones a La República, el especialista de Pacífico Business School,Jorge Carrillo Acosta, sostuvo que la presencia del banco enEE. UU. “ampliará la "oferta de servicios financieros y simplificará las transacc

Creación de tabla

In [ ]:
#Insertamos valores a la tabla

insert_query = f'''
    INSERT INTO NOTICIAS_CREDICORP (
    CONCEPT ,
    TITLE ,
    PUBLISHED_DATE ,
    URL ,
    CONTENT
    ) VALUES (
        :1, :2, :3, :4, :5
    )
'''

execute_query(insert_query, resultados_noticias_list)  # Ejecutar solo una vez porque la función es insert

Insert ejecutado en 0.50 segundos.


Ignorar duplicados automáticamente

In [ ]:
# Observar resumen 
q = f'''
    SELECT 
        CONCEPT,
        COUNT(CONCEPT) NUMERO_NOTICIAS
    FROM NOTICIAS_CREDICORP
    GROUP BY CONCEPT
    ORDER BY 1
    '''
execute_query(q)